# Self-Editing Agent with LangGraph — Guided Notebook

**Goal:** Build a graph-based Python coding agent that reads files, edits existing files, runs Python scripts, and loops between the model and tools.

Run cells from top to bottom.

> **Safety:** This agent can overwrite files and execute Python scripts. Test with sample files, keep backups, and never run untrusted code.

## Workflow

```text
START → agent → [tool calls?]
                   ├─ Yes → tools → agent (loop)
                   └─ No  → END
```

The model requests tool calls; `ToolNode` executes them. The tool results return to the model as part of message history.


## 1. Install dependencies

In [ ]:
%pip install -U langchain langgraph langchain-groq python-dotenv

## 2. Imports and what they do

- `sys`: current Python interpreter and command-line arguments.
- `os`: environment variables.
- `Annotated`: adds metadata to a type annotation; used here with a message reducer.
- `Literal`: limits a return type to specified values.
- `SystemMessage`, `HumanMessage`: structured messages with roles.
- `StateGraph`, `START`, `END`: build the graph and define entry/exit points.
- `add_messages`: merges message updates into the state history.
- `ToolNode`: executes tool calls requested by the model.
- `TypedDict`: defines the shape of graph state.
- `load_dotenv`: loads `.env` values.
- `init_chat_model`: initializes the chat model.
- `Path`: filesystem path handling.
- `InMemorySaver`: stores checkpoints in RAM.
- `subprocess`: runs scripts in a separate process.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from typing import Annotated, Literal

from dotenv import load_dotenv
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")


## 3. File tools

Tools are Python functions the model is allowed to request. The graph orchestrates; the functions do the actual work.

**Fix from the supplied code:** use `Path(file_input)`, not `Path(*file_input)`. Unpacking a string with `*` splits it into characters.


In [ ]:
def read_file(file_input: str) -> str:
    """Read the contents of an existing text file."""
    file_path = Path(file_input)
    if not file_path.exists():
        raise ValueError(f"File does not exist: {file_path}")
    if file_path.is_dir():
        raise ValueError(f"{file_path} is a directory. Pass an exact file path.")
    if not file_path.is_file():
        raise ValueError("Path is not a regular file.")
    return file_path.read_text(encoding="utf-8")


def write_file(file_input: str, content: str) -> str:
    """Replace the contents of an existing text file."""
    file_path = Path(file_input)
    if not file_path.exists():
        raise ValueError(f"File does not exist: {file_path}")
    if not file_path.is_file():
        raise ValueError("Path is not a regular file.")
    file_path.write_text(content, encoding="utf-8")
    return f"File updated successfully: {file_path}"


### `run_code`

`sys.executable` uses the notebook's Python interpreter. `subprocess.run` starts the script in another process; stdout and stderr are captured, with a 20-second timeout.

**Fix:** `[sys.executable, *script_path]` incorrectly spreads the path string into characters. Use `[sys.executable, str(file_path)]`.


In [ ]:
def run_code(script_path: str) -> str:
    """Run an existing Python script and return its output and errors."""
    file_path = Path(script_path)
    if not file_path.exists():
        raise ValueError(f"Script does not exist: {file_path}")
    if not file_path.is_file() or file_path.suffix.lower() != ".py":
        raise ValueError("Provide an existing .py file.")

    command = [sys.executable, str(file_path)]
    if file_path.name == "lg_tools.py":
        command.append("--test")

    try:
        result = subprocess.run(
            command, capture_output=True, text=True, timeout=20, check=False
        )
    except subprocess.TimeoutExpired:
        return "Execution timed out after 20 seconds."

    return (
        f"Exit code: {result.returncode}\n"
        f"Output:\n{result.stdout}\n"
        f"Errors:\n{result.stderr}"
    )


## 4. System prompt

A system prompt gives the model instructions about its role, behavior, and tasks. It guides the model's decisions but does not create graph connections.

A system prompt alone cannot guarantee security. Production applications should also use file access restrictions and a secure environment to run code.


In [ ]:
SYSTEM_PROMPT = """
You are a self-editing Python coding agent.

Project files:
- lg_tools.py: Contains the LangGraph workflow and tool/model setup.

Rules:
1. Always inspect the existing code before editing.
2. When modifying tools or the graph, read lg_tools.py first.
3. Never pass "." or a directory to read_file.
4. Always use an exact file path.
5. If the target path is unknown, ask the user.
6. Edit only the requested file.
7. After editing, run the modified code and inspect stdout/stderr.
8. Fix errors and rerun when appropriate.
9. Never claim success without execution evidence.
10. Do not make unrelated changes.
"""


## 5. Graph state — `TypedDict`, `Annotated`, `add_messages`

`TypedDict` describes the state shape. `messages` stores user messages, assistant responses, and tool results. `Annotated[list, add_messages]` tells LangGraph to merge new message updates into history rather than replacing the whole list.


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


## 6. Create model-callable tools

`@tool` converts each function into a tool schema. Names, type hints, and docstrings help the model understand available actions and arguments.

In [ ]:
@tool
def read_file_tool(file_path: str) -> str:
    """Read an existing text file. Provide an exact file path."""
    return read_file(file_path)


@tool
def write_file_tool(file_path: str, content: str) -> str:
    """Replace an existing text file with the supplied content."""
    return write_file(file_path, content)


@tool
def run_code_tool(script_path: str) -> str:
    """Run an existing Python script and return output and errors."""
    return run_code(script_path)


tools = [read_file_tool, write_file_tool, run_code_tool]
print("Tools:", [t.name for t in tools])


## 7. Build the LangGraph

- `init_chat_model`: initializes the Groq-hosted chat model.
- `bind_tools`: makes tool schemas available to the model.
- `agent_node`: prepends system instructions, invokes the model, and returns its response.
- `should_continue`: checks whether the latest assistant message requests tools.
- `StateGraph`: registers nodes and transitions.
- `add_node`: registers a node.
- `add_edge`: adds a fixed transition.
- `add_conditional_edges`: chooses the next route based on a function.
- `ToolNode`: runs tool calls.
- `compile`: creates the executable graph and attaches checkpointing.


In [ ]:
def build_graph():
    model = init_chat_model(
        "openai/gpt-oss-120b",
        model_provider="groq",
        api_key=api_key,
        temperature=0.2,
    )
    model_with_tools = model.bind_tools(tools)

    def agent_node(state: AgentState):
        messages = [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
        response = model_with_tools.invoke(messages)
        return {"messages": [response]}

    def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
        last_message = state["messages"][-1]
        if getattr(last_message, "tool_calls", None):
            return "tools"
        return END

    graph_builder = StateGraph(AgentState)
    graph_builder.add_node("agent", agent_node)
    graph_builder.add_node("tools", ToolNode(tools))
    graph_builder.add_edge(START, "agent")
    graph_builder.add_conditional_edges(
        "agent", should_continue, {"tools": "tools", END: END}
    )
    graph_builder.add_edge("tools", "agent")

    return graph_builder.compile(checkpointer=InMemorySaver())


agent = build_graph()
print("LangGraph compiled successfully.")


## 8. `InMemorySaver` and `thread_id`

`InMemorySaver()` keeps graph checkpoints in RAM. Reusing the same `thread_id` lets later calls continue the same thread while the kernel is alive. Restarting the kernel clears this in-memory state.

For durable production memory, use a supported persistent checkpointer and define suitable retention/access policies.


## 9. Run the agent — provide your own input

Run this cell. It waits at `input("You: ")`. Type a request and press Enter; type `exit` or `quit` to stop.

Example: `Read lg_tools.py and explain the graph nodes.`

Use exact paths relative to the notebook's working directory, or provide an absolute path.


In [ ]:
config = {"configurable": {"thread_id": "self-editing-agent"}}

print("Self-Editing Agent is ready! Type 'exit' to stop.\n")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        print("Agent stopped.")
        break
    if not user_input:
        continue

    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )
    print("\nAgent:", result["messages"][-1].content, "\n")


## 10. LangGraph modules — quick recap

| Module / concept | Purpose |
|---|---|
| `StateGraph` | Defines stateful nodes and transitions |
| `START` | Entry point |
| `END` | Terminal point |
| `TypedDict` | Describes graph state structure |
| `Annotated` | Adds metadata to a type annotation |
| `add_messages` | Merges message updates into conversation history |
| `SystemMessage` | Instructions sent to the model |
| `HumanMessage` | Represents user input |
| `ToolNode` | Executes requested tool calls |
| `bind_tools` | Exposes tool definitions to the model |
| `add_node` | Registers a node |
| `add_edge` | Adds a fixed transition |
| `add_conditional_edges` | Adds conditional routing |
| `compile` | Builds the runnable graph |
| `InMemorySaver` | Saves checkpoints in memory |
| `thread_id` | Identifies a conversation thread |
| `invoke` | Runs the graph for an input state |

### Important notes
1. `ToolNode` executes tools; `should_continue` decides where to route next.
2. The `tools → agent` edge creates the tool-calling loop.
3. `InMemorySaver` saves graph state, not your source files.
4. `write_file` replaces file contents. Keep backups and review edits.
5. This notebook demonstrates `StateGraph`; it is different from using LangChain's higher-level `create_agent` API.
